In [ ]:
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime, timedelta
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# بخش 1: تعریف تمام توابع تحلیل (3 تابع مجزا - خوشه‌بندی ژنراتور)
# ============================================================================

def analysis_1_generator_dbscan(file_path, output_filename):
    """
    تحلیل با DBSCAN برای ژنراتور - کد شماره 1
    شامل: خوشه‌بندی با DBSCAN + شاخص تخریب + وضعیت سلامت
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 1 (DBSCAN - ژنراتور)")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                      'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    eps = 0.5
    min_samples = 5
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        # پیش‌پردازش با smooth
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        # استانداردسازی
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        # مدل DBSCAN
        model = DBSCAN(eps=eps, min_samples=min_samples)
        cluster_labels = model.fit_predict(scaled_data)
        df_model['Behavior_Cluster'] = cluster_labels
        
        unique_clusters = set(cluster_labels)
        n_clusters = len([x for x in unique_clusters if x != -1])
        n_noise = sum(1 for x in cluster_labels if x == -1)
        print(f"   تعداد خوشه‌ها: {n_clusters}")
        print(f"   تعداد نقاط نویز: {n_noise:,} ({n_noise/len(cluster_labels)*100:.2f}%)")
        
        # محاسبه شاخص تخریب
        core_mask = np.zeros(len(scaled_data), dtype=bool)
        core_mask[model.core_sample_indices_] = True
        
        unique_clusters = set(cluster_labels) - {-1}
        cluster_centers = {}
        for cluster_id in unique_clusters:
            cluster_core_points = scaled_data[(cluster_labels == cluster_id) & core_mask]
            if len(cluster_core_points) > 0:
                cluster_centers[cluster_id] = np.mean(cluster_core_points, axis=0)
            else:
                cluster_all_points = scaled_data[cluster_labels == cluster_id]
                if len(cluster_all_points) > 0:
                    cluster_centers[cluster_id] = np.mean(cluster_all_points, axis=0)
                else:
                    cluster_centers[cluster_id] = np.zeros(scaled_data.shape[1])
        
        max_distance = 0
        degradation = np.zeros(len(scaled_data))
        
        for i in range(len(scaled_data)):
            cluster_id = cluster_labels[i]
            if cluster_id == -1:
                degradation[i] = 0
            else:
                if i in model.core_sample_indices_:
                    center = cluster_centers.get(cluster_id, np.zeros(scaled_data.shape[1]))
                    degradation[i] = np.linalg.norm(scaled_data[i] - center)
                else:
                    cluster_core_indices = [idx for idx in model.core_sample_indices_ 
                                           if cluster_labels[idx] == cluster_id]
                    if len(cluster_core_indices) > 0:
                        core_points = scaled_data[cluster_core_indices]
                        distances_to_cores = np.linalg.norm(core_points - scaled_data[i], axis=1)
                        degradation[i] = np.min(distances_to_cores)
                    else:
                        center = cluster_centers.get(cluster_id, np.zeros(scaled_data.shape[1]))
                        degradation[i] = np.linalg.norm(scaled_data[i] - center)
            if degradation[i] > max_distance:
                max_distance = degradation[i]
        
        for i in range(len(scaled_data)):
            if cluster_labels[i] == -1:
                degradation[i] = max_distance + 1.0
        
        df_model['Degradation_Index'] = degradation
        print(f"   محدوده شاخص تخریب: {df_model['Degradation_Index'].min():.4f} تا {df_model['Degradation_Index'].max():.4f}")
        
        # لیبل‌گذاری وضعیت سلامت
        def get_health_status(row):
            if row['Behavior_Cluster'] == -1:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 0.95:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 0.85:
                return "Observation Required (Pattern Change)"
            else:
                return "Healthy (Optimal Performance)"
        
        df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
        
        status_counts = df_model['Health_Status'].value_counts()
        print(f"\n📊 توزیع وضعیت‌ها:")
        for status, count in status_counts.items():
            print(f"   {status}: {count:,} ({count/len(df_model)*100:.2f}%)")
        
        # فیلتر ۳۰ روز آخر
        if 'date' in df_model.columns:
            last_dt = df_model['date'].max()
            start_date = last_dt - timedelta(days=30)
            final_df = df_model[df_model['date'] >= start_date].copy()
            print(f"\n📊 رکوردهای ۳۰ روز آخر: {len(final_df):,}")
        else:
            final_df = df_model
            print("   ⚠️ ستون 'date' وجود ندارد")
        
        # ذخیره خروجی
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_df.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_2_generator_lof(file_path, output_filename):
    """
    تحلیل با Local Outlier Factor (LOF) برای ژنراتور - کد شماره 2
    شامل: تشخیص ناهنجاری با LOF + شاخص تخریب + وضعیت سلامت
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 2 (LOF - ژنراتور)")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                      'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    n_neighbors = 20
    contamination = 0.05
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        # پیش‌پردازش با smooth
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        # استانداردسازی
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        # مدل LOF
        lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
        df_model['Behavior_Cluster'] = lof.fit_predict(scaled_data)
        
        lof_scores = -lof.negative_outlier_factor_
        df_model['Degradation_Index'] = lof_scores
        
        anomaly_count = (df_model['Behavior_Cluster'] == -1).sum()
        print(f"   تعداد ناهنجاری‌ها: {anomaly_count:,} ({anomaly_count/len(df_model)*100:.2f}%)")
        print(f"   محدوده شاخص تخریب: {df_model['Degradation_Index'].min():.4f} تا {df_model['Degradation_Index'].max():.4f}")
        
        # لیبل‌گذاری وضعیت سلامت
        def get_health_status(row):
            if row['Degradation_Index'] > 2.0 or row['Behavior_Cluster'] == -1:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 1.3:
                return "Observation Required (Pattern Change)"
            else:
                return "Healthy (Optimal Performance)"
        
        df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
        
        status_counts = df_model['Health_Status'].value_counts()
        print(f"\n📊 توزیع وضعیت‌ها:")
        for status, count in status_counts.items():
            print(f"   {status}: {count:,} ({count/len(df_model)*100:.2f}%)")
        
        # فیلتر ۳۰ روز آخر
        if 'date' in df_model.columns:
            last_date = df_model['date'].max()
            start_date = last_date - timedelta(days=30)
            final_output = df_model[df_model['date'] >= start_date].copy()
            print(f"\n📊 رکوردهای ۳۰ روز آخر: {len(final_output):,}")
        else:
            final_output = df_model
            print("   ⚠️ ستون 'date' وجود ندارد")
        
        # ذخیره خروجی
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_output.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_3_generator_kmeans(file_path, output_filename):
    """
    تحلیل با K-Means برای ژنراتور - کد شماره 3
    شامل: خوشه‌بندی با K-Means + شاخص تخریب + وضعیت سلامت
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 3 (K-Means - ژنراتور)")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                      'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    n_clusters = 3
    
    try:
        if not os.path.exists(file_path):
            print(f"❌ فایل یافت نشد: {file_path}")
            return False
        
        df = pd.read_excel(file_path)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        df_model = df.dropna(subset=target_sensors).copy()
        print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
        
        # پیش‌پردازش با smooth
        smooth_cols = []
        for sensor in target_sensors:
            name = f'{sensor}_smooth'
            df_model[name] = df_model[sensor].rolling(window=5, center=True).mean()
            smooth_cols.append(name)
        df_model = df_model.dropna(subset=smooth_cols).copy()
        print(f"✅ پیش‌پردازش انجام شد. تعداد رکوردها: {len(df_model):,}")
        
        # استانداردسازی
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_model[smooth_cols])
        
        # مدل K-Means
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df_model['Behavior_Cluster'] = kmeans.fit_predict(scaled_data)
        
        cluster_counts = df_model['Behavior_Cluster'].value_counts().sort_index()
        print(f"   تعداد خوشه‌ها: {len(cluster_counts)}")
        for cluster_id, count in cluster_counts.items():
            print(f"   خوشه {cluster_id}: {count:,} رکورد ({count/len(df_model)*100:.2f}%)")
        
        # شاخص تخریب
        distances = np.linalg.norm(scaled_data - kmeans.cluster_centers_[df_model['Behavior_Cluster']], axis=1)
        df_model['Degradation_Index'] = distances
        print(f"   محدوده شاخص تخریب: {df_model['Degradation_Index'].min():.4f} تا {df_model['Degradation_Index'].max():.4f}")
        
        # لیبل‌گذاری وضعیت سلامت
        def get_health_status(row):
            if row['Degradation_Index'] > 2.5:
                return "Investigation Needed (Operational Drift)"
            elif row['Degradation_Index'] > 1.5:
                return "Observation Required (Pattern Change)"
            else:
                return "Healthy (Optimal Performance)"
        
        df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
        
        status_counts = df_model['Health_Status'].value_counts()
        print(f"\n📊 توزیع وضعیت‌ها:")
        for status, count in status_counts.items():
            print(f"   {status}: {count:,} ({count/len(df_model)*100:.2f}%)")
        
        # فیلتر ۳۰ روز آخر
        if 'date' in df_model.columns:
            last_date = df_model['date'].max()
            one_month_ago = last_date - timedelta(days=30)
            final_output = df_model[df_model['date'] >= one_month_ago].copy()
            print(f"\n📊 رکوردهای ۳۰ روز آخر: {len(final_output):,}")
        else:
            final_output = df_model
            print("   ⚠️ ستون 'date' وجود ندارد")
        
        # ذخیره خروجی
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_output.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# ============================================================================
# بخش 2: تعریف وظایف (Jobs) - 3 وظیفه خوشه‌بندی ژنراتور
# ============================================================================

def get_analysis_jobs():
    """
    تعریف ۳ وظیفه تحلیل با مسیرهای ورودی و خروجی مربوطه - خوشه‌بندی ژنراتور
    """
    jobs = [
        {
            'name': 'Analysis 1 - DBSCAN (Generator)',
            'function': analysis_1_generator_dbscan,
            'file_path': r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx',
            'output_filename': r'outputs\G11\dsas_g11_generator_bearings_clustering\clustering\dsas_g11_generator_bearings\clustering_g11_generator_bearings_output2.xlsx'
        },
        {
            'name': 'Analysis 2 - LOF (Generator)',
            'function': analysis_2_generator_lof,
            'file_path': r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx',
            'output_filename': r'outputs\G11\dsas_g11_generator_bearings_clustering\clustering\dsas_g11_generator_bearings\clustering_g11_generator_bearings_output3.xlsx'
        },
        {
            'name': 'Analysis 3 - K-Means (Generator)',
            'function': analysis_3_generator_kmeans,
            'file_path': r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx',
            'output_filename': r'outputs\G11\dsas_g11_generator_bearings_clustering\clustering\dsas_g11_generator_bearings\clustering_g11_generator_bearings_output4.xlsx'
        }
    ]
    return jobs


def run_all_analyses():
    """
    اجرای تمام ۳ تحلیل به ترتیب
    """
    print("\n" + "="*80)
    print(f"🚀 شروع اجرای همه تحلیل‌ها در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    print("📋 لیست تحلیلها:")
    print("   1. DBSCAN - خوشه‌بندی مبتنی بر چگالی")
    print("   2. Local Outlier Factor (LOF) - تشخیص ناهنجاری محلی")
    print("   3. K-Means - خوشه‌بندی")
    print("="*80)
    
    jobs = get_analysis_jobs()
    results = []
    
    for i, job in enumerate(jobs, 1):
        print(f"\n{'#'*80}")
        print(f"# اجرای وظیفه {i} از {len(jobs)}: {job['name']}")
        print(f"{'#'*80}")
        
        try:
            success = job['function'](job['file_path'], job['output_filename'])
            results.append({
                'job_name': job['name'],
                'success': success,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
            
            if success:
                print(f"✅ وظیفه {i} با موفقیت کامل شد")
            else:
                print(f"❌ وظیفه {i} با شکست مواجه شد")
                
        except Exception as e:
            print(f"❌ خطای غیرمنتظره در وظیفه {i}: {e}")
            results.append({
                'job_name': job['name'],
                'success': False,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'error': str(e)
            })
    
    # گزارش نهایی
    print("\n" + "="*80)
    print("📊 گزارش نهایی اجرای همه تحلیل‌ها")
    print("="*80)
    
    success_count = sum(1 for r in results if r['success'])
    total_count = len(results)
    
    print(f"✅ موفق: {success_count} از {total_count}")
    print(f"❌ ناموفق: {total_count - success_count} از {total_count}")
    
    for r in results:
        status = "✅" if r['success'] else "❌"
        print(f"   {status} {r['job_name']} - {r['timestamp']}")
    
    print("="*80)
    print(f"🏁 پایان اجرا در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    return results


# ============================================================================
# بخش 3: زمان‌بندی (Scheduler) با دو زمان ۹:۰۰ و ۲۱:۰۰
# ============================================================================

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز ساعت ۹:۰۰ و ۲۱:۰۰)
    """
    print("="*80)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل خوشه‌بندی ژنراتور")
    print("="*80)
    print("📋 شامل ۳ الگوریتم خوشه‌بندی:")
    print("   1. DBSCAN")
    print("   2. Local Outlier Factor (LOF)")
    print("   3. K-Means")
    print("="*80)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 09:00")
    print("   - ساعت 21:00")
    print("="*80)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*80)
    
    last_run_times = {}  # ذخیره زمان‌های اجرا شده
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص (۹:۰۰ و ۲۱:۰۰)
            if current_time in ["12:25", "12:35"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_times.get(current_time) != now.strftime("%Y-%m-%d"):
                    print("\n" + "="*80)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*80)
                    
                    # اجرای همه تحلیل‌ها
                    results = run_all_analyses()
                    
                    # ثبت زمان اجرا
                    last_run_times[current_time] = now.strftime("%Y-%m-%d")
                    
                    print("\n" + "="*80)
                    print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                    print("="*80)
                    
                    # ۶۰ ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(60)
            
            # هر ۳۰ ثانیه یکبار بررسی کن
            time.sleep(30)
            
        except KeyboardInterrupt:
            print("\n" + "="*80)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*80)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)


# ============================================================================
# بخش 4: اجرای اصلی
# ============================================================================

if __name__ == "__main__":
    try:
        print("="*80)
        print("🚀 شروع برنامه جامع تحلیل خوشه‌بندی ژنراتور (۳ الگوریتم یکپارچه)")
        print("="*80)
        print("📋 لیست الگوریتم‌ها:")
        print("   1. DBSCAN - خوشه‌بندی مبتنی بر چگالی")
        print("   2. Local Outlier Factor (LOF) - تشخیص ناهنجاری محلی")
        print("   3. K-Means - خوشه‌بندی با میانگین‌گیری")
        print("="*80)
        print("⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00")
        print("="*80)
        
        # اجرای زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        import traceback
        traceback.print_exc()
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه جامع تحلیل خوشه‌بندی ژنراتور (۳ الگوریتم یکپارچه)
📋 لیست الگوریتم‌ها:
   1. DBSCAN - خوشه‌بندی مبتنی بر چگالی
   2. Local Outlier Factor (LOF) - تشخیص ناهنجاری محلی
   3. K-Means - خوشه‌بندی با میانگین‌گیری
⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل خوشه‌بندی ژنراتور
📋 شامل ۳ الگوریتم خوشه‌بندی:
   1. DBSCAN
   2. Local Outlier Factor (LOF)
   3. K-Means
⏰ زمان‌های اجرا (هر روز):
   - ساعت 09:00
   - ساعت 21:00
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-09 12:25:11

🚀 شروع اجرای همه تحلیل‌ها در 2026-07-09 12:25:11
📋 لیست تحلیلها:
   1. DBSCAN - خوشه‌بندی مبتنی بر چگالی
   2. Local Outlier Factor (LOF) - تشخیص ناهنجاری محلی
   3. K-Means - خوشه‌بندی

################################################################################
# اجرای وظیفه 1 از 3: Analysis 1 - DBSCAN (Generator)
################################################################################

🔄 شروع تحلیل شماره 1 (DBSCAN - ژنراتور)
✅